# Chapter 5 — Minimal VLM Training (Projector Only)

This notebook performs a **single-step multimodal training loop**.

Goal:
- freeze language model
- train vision → LM projector
- verify loss + backward pass

This is a sanity check, not a benchmark.

## Training Strategy

We train **only the projector**.

Frozen:
- vision encoder
- language model

Trainable:
- projector parameters

Supervision:
- text-only next-token prediction

## What This Notebook Proves

✔ forward pass works with gradients  
✔ loss decreases (at least numerically)  
✔ gradients flow into projector  
✔ LM weights remain frozen  

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Dimensions & Hyperparameters

In [2]:
BATCH = 2
NUM_PATCHES = 64
TEXT_LEN = 8

VISION_DIM = 512
LM_DIM = 1024
VOCAB_SIZE = 10_000

LR = 1e-3

## Tiny Multimodal Dataset

We create a synthetic dataset:
- random images
- random token IDs
- next-token labels

This isolates plumbing from data quality.

In [3]:
images = torch.randn(BATCH, 3, 224, 224)

input_ids = torch.randint(0, VOCAB_SIZE, (BATCH, TEXT_LEN))
labels = input_ids.clone()
labels[:, :-1] = input_ids[:, 1:]
labels[:, -1] = -100  # ignore last token

## Model Components

- Dummy vision encoder (frozen)
- Trainable projector
- Dummy LM (frozen)

In [4]:
class DummyVisionEncoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.out_dim = out_dim

    def forward(self, x):
        B = x.shape[0]
        return torch.randn(B, NUM_PATCHES, self.out_dim)

vision_encoder = DummyVisionEncoder(VISION_DIM)
for p in vision_encoder.parameters():
    p.requires_grad = False

In [5]:
class LinearProjector(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)

projector = LinearProjector(VISION_DIM, LM_DIM)

In [6]:
class DummyLM(nn.Module):
    def __init__(self, dim, vocab):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.lm_head = nn.Linear(dim, vocab)

    def forward(self, embeds):
        return self.lm_head(embeds)

lm = DummyLM(LM_DIM, VOCAB_SIZE)
for p in lm.parameters():
    p.requires_grad = False

## Text Embeddings

We embed tokens but do not train the embedding table.

In [7]:
text_embeds = lm.embed(input_ids)

## Vision → Projector Path

In [8]:
vision_feats = vision_encoder(images)
vision_embeds = projector(vision_feats)

## Multimodal Fusion (Prepend)

In [9]:
im_start = torch.zeros(BATCH, 1, LM_DIM)
im_end   = torch.zeros(BATCH, 1, LM_DIM)

image_block = torch.cat([im_start, vision_embeds, im_end], dim=1)
joint_embeds = torch.cat([image_block, text_embeds], dim=1)

image_len = image_block.shape[1]

## Forward Pass

In [10]:
logits = lm(joint_embeds)
text_logits = logits[:, image_len:, :]

## Text-Only Loss

We supervise only text positions.

In [11]:
loss = F.cross_entropy(
    text_logits.reshape(-1, VOCAB_SIZE),
    labels.reshape(-1),
    ignore_index=-100
)

print("Loss:", loss.item())

Loss: 9.455902099609375


## Backward Pass

In [12]:
optimizer = torch.optim.AdamW(projector.parameters(), lr=LR)

optimizer.zero_grad()
loss.backward()
optimizer.step()

## Gradient Sanity Checks

In [ ]:
proj_grad_norm = projector.proj.weight.grad.norm().item()
print("Projector grad norm:", proj_grad_norm)

assert proj_grad_norm > 0, "No gradient flowed to projector"

In [ ]:
for p in lm.parameters():
    assert p.grad is None

## What This Confirms

✔ multimodal loss is computable  
✔ gradients flow correctly  
✔ LM is frozen  
✔ projector is trainable  

You now have a *trainable VLM skeleton*.